# Cats vs Dogs Binary Classification

## 1. Data Loading and Generators

This project builds a binary image classifier to distinguish cats from dogs. We use image generators, data augmentation, a CNN, model evaluation, inference, and experiment comparison.


In [ ]:
# Prefilled block from the exercise
import os, math, re, random, json
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

DATA_ROOT = Path("data/cats_dogs")

train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir  = (DATA_ROOT / "test" / "test") if (DATA_ROOT / "test" / "test").exists() else (DATA_ROOT / "test")

IMG_HEIGHT = 180
IMG_WIDTH = 180
batch_size = 32
seed = 1337

def build_df_from_folder(folder: Path, labeled=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))

    rows = []

    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()

            if parent in {'cat','cats'}:
                label = 'cat'
            elif parent in {'dog','dogs'}:
                label = 'dog'
            else:
                if 'cat' in name:
                    label = 'cat'
                elif 'dog' in name:
                    label = 'dog'
                else:
                    continue

            rows.append({'filepath':f,'label':label})
        else:
            rows.append({'filepath':f})

    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full = build_df_from_folder(test_dir, labeled=False)

from sklearn.model_selection import train_test_split

df_tr, df_val = train_test_split(
    df_train_full,
    test_size=0.2,
    stratify=df_train_full['label'],
    random_state=seed
)


## 2. Data Inspection

In [ ]:
# Class distribution
print(df_tr['label'].value_counts())

sns.countplot(data=df_tr, x='label')
plt.title('Class Distribution')
plt.show()

# Comment:
# If counts are similar, the dataset is balanced.
# Otherwise, class weights may be needed.


In [ ]:
# Sample images
from tensorflow.keras.utils import load_img

fig, axes = plt.subplots(2,4, figsize=(12,6))

sample_df = df_tr.sample(8, random_state=42)

for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
    img = load_img(row['filepath'], target_size=(128,128))
    ax.imshow(img)
    ax.set_title(row['label'])
    ax.axis('off')

plt.tight_layout()
plt.show()


### Data Report

Potential sources of variability:

- Different poses of cats and dogs.
- Different backgrounds.
- Lighting conditions.
- Scale and camera angles.

These variations justify using data augmentation.


In [ ]:
# Augmented generators

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr,
    x_col='filepath',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary',
    batch_size=batch_size,
    shuffle=True
)

val_flow = val_gen.flow_from_dataframe(
    df_val,
    x_col='filepath',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary',
    batch_size=batch_size,
    shuffle=False
)

test_flow = test_gen.flow_from_dataframe(
    df_test_full,
    x_col='filepath',
    y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None,
    shuffle=False
)


## 3. Model Architecture

Architecture:

- 3 convolutional blocks.
- MaxPooling after each convolution.
- Dropout to reduce overfitting.
- Dense layer for learning higher-level patterns.
- Sigmoid output for binary classification.


In [ ]:
model = models.Sequential([

    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(1, activation='sigmoid')
])

model.summary()


## 4. Optimization Setup

- Optimizer: Adam
- Learning rate: default Adam value
- Batch size: 32
- EarlyStopping on validation loss
- ReduceLROnPlateau for adaptive learning rate


In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2
)


## 5. Training

In [ ]:
history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)


In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Accuracy')
plt.legend(['Train','Validation'])

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss')
plt.legend(['Train','Validation'])

plt.show()


Interpretation:

- If training accuracy increases while validation accuracy stagnates, overfitting occurs.
- Data augmentation and dropout help reduce this gap.


## 6. Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

loss, acc = model.evaluate(val_flow)

print('Validation Loss:', loss)
print('Validation Accuracy:', acc)

preds = model.predict(val_flow)
preds = (preds > 0.5).astype(int)

cm = confusion_matrix(
    val_flow.classes,
    preds[:len(val_flow.classes)]
)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Confusion Matrix')
plt.show()

print(classification_report(
    val_flow.classes,
    preds[:len(val_flow.classes)]
))


Explain which class is more frequently misclassified and discuss whether stronger augmentation or threshold tuning may help.


## 7. Inference on Unlabeled Test Set

In [ ]:
prob_dog = model.predict(test_flow).flatten()

submission = pd.DataFrame({
    'filepath': df_test_full['filepath'],
    'prob_dog': prob_dog,
    'pred_label': np.where(prob_dog >= 0.5, 'dog', 'cat')
})

submission.to_csv('test_predictions.csv', index=False)

submission.head()


## 8. Baseline vs Augmentation

In [ ]:
# Baseline without augmentation

baseline_gen = ImageDataGenerator(rescale=1./255)

baseline_train = baseline_gen.flow_from_dataframe(
    df_tr,
    x_col='filepath',
    y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary',
    batch_size=batch_size
)


Train the same architecture with the baseline generator and compare:

- Validation accuracy
- Validation loss
- Generalization gap

Expected result: augmentation should improve generalization.


## 9. Class Imbalance Handling

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(df_tr['label'])

weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=df_tr['label']
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)


If the dataset is imbalanced, retrain using:

```python
model.fit(..., class_weight=class_weights)
```

Then compare precision and recall.


## 10. Save Artifacts

In [ ]:
model.save('cats_dogs_model.keras')

config = {
    'img_height': IMG_HEIGHT,
    'img_width': IMG_WIDTH,
    'batch_size': batch_size,
    'optimizer': 'adam'
}

with open('training_config.json','w') as f:
    json.dump(config, f, indent=4)


Saving both model weights and configuration is essential for reproducibility.


## 11. Extension Proposal

### Transfer Learning with MobileNetV2

A lightweight pretrained backbone can significantly improve accuracy because low-level visual features such as edges, textures, and shapes are already learned from large image datasets.


## 12. Deliverables Checklist

- [x] Data report
- [x] Sample image grid
- [x] CNN description
- [x] Optimization rationale
- [x] Training curves
- [x] Confusion matrix
- [x] Precision / Recall
- [x] Test predictions CSV
- [x] Saved model
- [x] Training configuration file
